# Coverage gap solution — flattened reproduction notebook

Generated from the frozen scoring logic (`scoring/v1/checksum.txt` records the exact hashes). No project-local imports: every function below is the real `src/gaps.py` source, inlined verbatim by `scripts/submission/build_submission_notebook.py`. Do not hand-edit this file — regenerate it instead.

In [1]:
"""
GENERATED by scripts/submission/build_submission_notebook.py from the frozen src/ modules -- do not hand-edit. Regenerate if src/gaps.py (or anything it imports) changes. Uses only public packages (pandas); no project-local imports.
"""
from __future__ import annotations

from pathlib import Path

import pandas as pd

# ============================================================================
# from src/config.py
# ============================================================================
REGIONS = ["maricopa-az", "northern-ca", "eastern-ok", "south-central-tx"]

# ============================================================================
# from src/schemas.py
# ============================================================================
_GAP_RATIO_COLUMNS = (
    "transport_gap",
    "building_gap_centroid",
    "building_gap_intersection",
    "poi_gap_fire",
    "poi_gap_ems",
    "poi_gap_schools",
    "poi_gap_establishments",
)

_DEFINED_COLUMN_NAMES = (
    "transport_defined",
    "building_gap_centroid_defined",
    "building_gap_intersection_defined",
    "poi_gap_fire_defined",
    "poi_gap_ems_defined",
    "poi_gap_schools_defined",
    "poi_gap_establishments_defined",
)

COMPETITION_ALLOWED_COLUMNS: frozenset[str] = frozenset(_GAP_RATIO_COLUMNS + _DEFINED_COLUMN_NAMES)

# ============================================================================
# from src/features.py
# ============================================================================
def assert_competition_only(columns_used: set[str]) -> None:
    """Security-and-hardening framing, not a formality: an ALLOWLIST check, FAIL-CLOSED, least-
    privilege by default.

    `src.schemas.COMPETITION_ALLOWED_COLUMNS` is the single frozen source of truth for which
    column names a caller (Stage 7's `build_submission.py`, once it wires this in) may read with
    `feature_role=competition`. This function does not itself decide which columns are allowed â€”
    it only enforces the frozen set â€” so a call site can never widen its own permissions by
    passing a bigger set; the set lives in one place, reviewed as a real code change, not as a
    parameter any caller controls.

    ALLOWLIST, not a blocklist: `columns_used` is checked against what IS permitted, not against a
    list of what's forbidden â€” a brand-new strata column added to `STRATA_FEATURE_COLUMNS` next
    week is rejected by default (safe) rather than silently passing because nobody thought to
    blocklist it yet (unsafe). FAIL-CLOSED: `COMPETITION_ALLOWED_COLUMNS` is empty today (Step 0
    Ambiguity 2 â€” the winning gap-value variant isn't frozen until Stage 7's calibration), so this
    currently raises on ANY non-empty `columns_used`, by design â€” a caller that trips this before
    Stage 7 widens the set has found a real bug (something is reading a not-yet-frozen or
    strata-sourced column into the score), not a false alarm to silence.

    Raises `ValueError` naming every disallowed column, rather than returning a bool a caller could
    forget to check â€” a permission gate that can be silently ignored is not a permission gate."""
    # Imported here, not at module level: src.schemas imports STRATA_FEATURE_COLUMNS from this
    # module (Step 6), so a top-level `from src.schemas import ...` here would be a circular
    # import. This function is the only thing in this module that needs schemas.py.

    disallowed = set(columns_used) - COMPETITION_ALLOWED_COLUMNS
    if disallowed:
        raise ValueError(
            f"assert_competition_only: {sorted(disallowed)!r} not in COMPETITION_ALLOWED_COLUMNS "
            f"(src/schemas.py) â€” these columns are not permitted to influence the scored "
            "coverage-gap number. If this is a genuine, reviewed addition to the competition-"
            "scored column set (e.g. Stage 7 freezing a winning gap-value variant), widen "
            "COMPETITION_ALLOWED_COLUMNS itself; do not work around this check at the call site."
        )

# ============================================================================
# from src/gaps.py
# ============================================================================
PROCESSED_DIR = Path("data/processed")

TRANSPORT_GAP_COLUMN = "transport_gap"

TRANSPORT_DEFINED_COLUMN = "transport_defined"

BUILDING_GAP_COLUMN = "building_gap_centroid"  # Stage 7 Step 6 Tier A calibration result: centroid

POI_SUBPART_COLUMNS = ("poi_gap_fire", "poi_gap_ems", "poi_gap_schools", "poi_gap_establishments")

HIFLD_SUBPART_COLUMNS = POI_SUBPART_COLUMNS[:3]

CBP_COLUMN = POI_SUBPART_COLUMNS[3]

def capped_ratio_gap(overture: float, reference: float) -> float | None:
    """The challenge's capped-ratio gap formula: ``1 - min(1, overture / reference)``.

    Returns ``None`` (not ``0.0``) when ``reference == 0`` â€” there is nothing to compare against,
    so the value is undefined, not "fully covered". This is R-005 from the risk register: an
    undefined component must be excluded from `coverage_gap_score`'s mean entirely, never silently
    zeroed. Callers must not coerce this ``None`` into a number.

    The ratio is capped at 1 before subtracting, so a tract where Overture reports MORE than the
    reference (overture > reference) still floors at gap 0.0 â€” never negative.

    Not called from `score_region` below: Stage 6 already applied this exact formula when it
    materialized `transport_gap`/`building_gap_*`/`poi_gap_*` (src/schemas.py's `_GAP_RATIO_COLUMNS`
    comment: "every one is a `_capped_ratio` output"). This function exists so
    `tests/test_gap_arithmetic.py` can lock down the formula's edge cases independently of Stage 6's
    own implementation, and so a human reader has one place that states the formula in code."""
    if reference == 0:
        return None
    return 1 - min(1.0, overture / reference)

def coverage_gap_score(components: dict[str, float | None]) -> float | None:
    """The composite score: the arithmetic mean of only the DEFINED component gap values â€”
    ``None`` or NaN both count as undefined (``pd.notna`` covers real pandas data, which stores
    "undefined" as NaN, not the ``None`` this function's own unit tests use). Divisor is however
    many components are defined (1, 2, or 3) â€” never a fixed 3, and an undefined component is
    excluded, not treated as 0. Returns ``None`` if every component is undefined for that tract."""
    defined = [v for v in components.values() if pd.notna(v)]
    if not defined:
        return None
    return sum(defined) / len(defined)

def _poi_gap_for_row(row: pd.Series) -> tuple[float | None, bool]:
    """`poi_gap`'s NESTED half-definedness rule, per the challenge README (see
    `HIFLD_SUBPART_COLUMNS`'s comment above for the exact quote): `poi_gap_hifld` is the mean of
    whichever of {fire, ems, schools} are defined for this tract; `poi_gap` is then the mean of
    `poi_gap_hifld` and the CBP-establishments value, with either half excluded from that outer
    mean if it's undefined. This is NOT a flat mean of all four sub-parts -- that flat-mean
    version was this function's original, incorrect implementation (caught via source-driven
    verification against the README, not a test or a leaderboard result -- see
    `tests/test_gap_arithmetic.py`'s `test_poi_gap_uses_nested_mean_not_flat_mean_...` for the
    regression test this fix must pass). Uses each sub-part's own `<col>_defined` boolean, not a
    bare `notna()` check on the value (Stage 6's placeholder convention stores a numeric 0 even
    for undefined sub-parts, per the real sample-submission template: `poi_gap_ems=0` with
    `poi_defined_ems=FALSE` is a real row in `SampleSubmission 3.csv`).

    Returns `(poi_gap, poi_defined)` -- `poi_defined` is True iff at least one of the two halves
    is defined, matching the template's own separate `poi_defined` column (distinct from
    `poi_defined_fire`/`poi_defined_ems`/`poi_defined_schools`/`poi_defined_cbp`)."""
    hifld_defined_values = [
        row[col] for col in HIFLD_SUBPART_COLUMNS if bool(row[f"{col}_defined"])
    ]
    poi_gap_hifld = (
        sum(hifld_defined_values) / len(hifld_defined_values) if hifld_defined_values else None
    )

    poi_gap_cbp = row[CBP_COLUMN] if bool(row[f"{CBP_COLUMN}_defined"]) else None

    halves = [v for v in (poi_gap_hifld, poi_gap_cbp) if pd.notna(v)]
    if not halves:
        return None, False
    return sum(halves) / len(halves), True

def score_region(
    tract_features: pd.DataFrame, *, building_gap_column: str = BUILDING_GAP_COLUMN
) -> pd.DataFrame:
    """Scores one region's Stage 6 tract-features table. Returns a tidy per-tract table:
    `GEOID`, `region`, each component gap + its `_defined` flag, and `coverage_gap_score`.

    `building_gap_column` -- Stage 7 Step 6 (Tier A calibration): which of the two Stage-6
    building-assignment candidates (`building_gap_centroid` or `building_gap_intersection`) to
    use for THIS call. Defaults to `BUILDING_GAP_COLUMN` (the current best-trusted default, not
    yet frozen) so every existing caller is unaffected; `scripts/tier_a_calibration.py` is the
    only caller that passes a non-default value, one region at a time, per the designed-experiment
    protocol (three regions held fixed, one varied). Both candidates are already in
    `src.schemas.COMPETITION_ALLOWED_COLUMNS` -- no schema change needed to support this.

    Calls `assert_competition_only` first, naming every Stage-6 column this function reads -- the
    fail-closed gate raises before any real computation if a column isn't in
    `src.schemas.COMPETITION_ALLOWED_COLUMNS`, catching a typo'd or not-yet-allowed column name
    before it can influence a score."""
    columns_used = {
        TRANSPORT_GAP_COLUMN,
        TRANSPORT_DEFINED_COLUMN,
        building_gap_column,
        f"{building_gap_column}_defined",
        *POI_SUBPART_COLUMNS,
        *(f"{c}_defined" for c in POI_SUBPART_COLUMNS),
    }
    assert_competition_only(columns_used)

    out = pd.DataFrame(
        {"GEOID": tract_features["GEOID"], "region": tract_features["region"]}
    )
    out["transport_gap"] = tract_features[TRANSPORT_GAP_COLUMN]
    out["transport_defined"] = tract_features[TRANSPORT_DEFINED_COLUMN]
    out["building_gap"] = tract_features[building_gap_column]
    out["building_defined"] = tract_features[f"{building_gap_column}_defined"]

    poi_results = tract_features.apply(_poi_gap_for_row, axis=1, result_type="expand")
    out["poi_gap"] = poi_results[0]
    out["poi_defined"] = poi_results[1]
    # Also carry the four POI sub-part columns through -- already in COMPETITION_ALLOWED_COLUMNS
    # and already read above (columns_used), Step 5's build_submission.py needs them as their own
    # template columns (poi_gap_fire/ems/schools/cbp), not just folded into the poi_gap mean.
    for col in POI_SUBPART_COLUMNS:
        out[col] = tract_features[col]
        out[f"{col}_defined"] = tract_features[f"{col}_defined"]

    out["coverage_gap_score"] = out.apply(
        lambda r: coverage_gap_score(
            {
                "transport_gap": r["transport_gap"],
                "building_gap": r["building_gap"],
                "poi_gap": r["poi_gap"],
            }
        ),
        axis=1,
    )
    return out

def score_all_regions(
    regions: list[str] = REGIONS,
    *,
    building_gap_overrides: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Reads every region's `data/processed/<region>-tract-features.parquet` and returns one
    concatenated scored table across all four regions.

    `building_gap_overrides` -- Stage 7 Step 6 Tier A calibration: an optional `{region:
    building_gap_column}` mapping. A region present in the mapping is scored with that building-
    assignment column instead of `BUILDING_GAP_COLUMN`'s default; every region absent from the
    mapping (or when the mapping itself is omitted) keeps the default -- this is exactly the
    "three regions held fixed, one varied" shape the designed-experiment protocol needs, and the
    "flip every region at once" cross-region-confirmation shape (pass all four regions in the
    mapping) with the same parameter."""
    overrides = building_gap_overrides or {}
    frames = []
    for region in regions:
        path = PROCESSED_DIR / f"{region}-tract-features.parquet"
        tract_features = pd.read_parquet(path)
        building_gap_column = overrides.get(region, BUILDING_GAP_COLUMN)
        frames.append(score_region(tract_features, building_gap_column=building_gap_column))
    return pd.concat(frames, ignore_index=True)


## Self-check: Step 2 golden-case fixtures, reproduced here

In [2]:
# --- self-check: re-run Step 2's golden-case fixtures against the flattened functions above ---
assert capped_ratio_gap(overture=0, reference=100) == 1.0
assert capped_ratio_gap(overture=100, reference=100) == 0.0
assert capped_ratio_gap(overture=150, reference=100) == 0.0  # caps at 0, never negative
assert capped_ratio_gap(overture=0, reference=0) is None      # undefined, never zeroed (R-005)
assert capped_ratio_gap(overture=30, reference=40) == 0.25

assert coverage_gap_score({"transport_gap": 0.25, "building_gap": 0.75, "poi_gap": None}) == 0.5
assert abs(coverage_gap_score({"transport_gap": 0.2, "building_gap": 0.4, "poi_gap": 0.6}) - 0.4) < 1e-9
assert coverage_gap_score({"transport_gap": None, "building_gap": None, "poi_gap": None}) is None

_synthetic = pd.DataFrame([{
    "GEOID": "12345678900",
    "region": "eastern-ok",
    "transport_gap": 0.1, "transport_defined": True,
    "building_gap_centroid": 0.3, "building_gap_centroid_defined": True,
    "building_gap_intersection": 0.9, "building_gap_intersection_defined": True,
    "poi_gap_fire": 0.5, "poi_gap_fire_defined": True,
    "poi_gap_ems": 0.5, "poi_gap_ems_defined": True,
    "poi_gap_schools": 0.5, "poi_gap_schools_defined": True,
    "poi_gap_establishments": 0.5, "poi_gap_establishments_defined": True,
}])
_scored = score_region(_synthetic)
assert abs(_scored.loc[0, "building_gap"] - 0.3) < 1e-9  # frozen default: centroid, not intersection
_scored_override = score_region(_synthetic, building_gap_column="building_gap_intersection")
assert abs(_scored_override.loc[0, "building_gap"] - 0.9) < 1e-9

_expected_output_columns = {
    "GEOID", "region", "transport_gap", "transport_defined", "building_gap", "building_defined",
    "poi_gap", "poi_defined", "poi_gap_fire", "poi_gap_fire_defined", "poi_gap_ems",
    "poi_gap_ems_defined", "poi_gap_schools", "poi_gap_schools_defined",
    "poi_gap_establishments", "poi_gap_establishments_defined", "coverage_gap_score",
}
assert set(_scored.columns) == _expected_output_columns, (
    f"score_region output schema drifted: {set(_scored.columns) ^ _expected_output_columns}"
)
print("Flattened notebook self-check: PASS (Step 2 fixtures reproduced, output schema confirmed).")


Flattened notebook self-check: PASS (Step 2 fixtures reproduced, output schema confirmed).


## Run against the real Stage 6 feature tables

Requires `data/processed/<region>-tract-features.parquet` (produced by Stage 6, not part of this notebook) on the local filesystem at `data/processed/`.

In [3]:
# PROCESSED_DIR above is relative ("data/processed") -- Jupyter's default working
# directory is wherever this .ipynb file sits (submission/), not the repo root, so
# find and switch to the repo root first rather than requiring the user to launch
# Jupyter from a specific directory.
import os

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        "No data/processed/ directory found walking up from the current working "
        "directory -- run this notebook from inside the project repo."
    )

os.chdir(_find_repo_root(Path.cwd()))
print(f"Working directory: {Path.cwd()}")

scored = score_all_regions()
print(f"Scored {len(scored)} tracts across {scored['region'].nunique()} regions.")
scored.head()

Working directory: d:\PROJECTS\bias-bounty-mapping-equity
Scored 9386 tracts across 4 regions.


,GEOID,region,transport_gap,transport_defined,building_gap,building_defined,poi_gap,poi_defined,poi_gap_fire,poi_gap_fire_defined,poi_gap_ems,poi_gap_ems_defined,poi_gap_schools,poi_gap_schools_defined,poi_gap_establishments,poi_gap_establishments_defined,coverage_gap_score
0,04027980004,maricopa-az,0.003037,True,0.0,True,0.000,True,NaN,False,NaN,False,NaN,False,0.0,True,0.001012
1,04027980003,maricopa-az,NaN,False,0.0,True,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,0.000000
2,04019940800,maricopa-az,0.284211,True,0.0,True,0.250,True,1.0,True,NaN,False,0.0,True,0.0,True,0.178070
3,04019004425,maricopa-az,NaN,False,0.0,True,0.125,True,0.5,True,NaN,False,0.0,True,0.0,True,0.062500
4,04013940700,maricopa-az,NaN,False,0.0,True,NaN,False,NaN,False,NaN,False,NaN,False,NaN,False,0.000000
